<div class="alert alert-danger" role="alert">
    <div class="row vertical-align">
        <div class="col-xs-1 text-center">
            <i class="fa fa-exclamation-triangle fa-2x"></i>
        </div>
        <div class="col-xs-11">
                <strong>July 14, 2021</strong>                   
        </div>   
    </div> 
</div>

In [1]:
import os
import glob
import random
import folium
import numpy as np
import pandas as pd
import wbgapi as wb
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

import ee
ee.Initialize()
import leafmap
#leafmap.update_package()
import geemap
#geemap.update_package()

# AG.LND.EL5M.ZS
#https://towardsdatascience.com/choropleth-maps-101-using-plotly-5daf85e7275d

# Define directory

In [2]:
out_dir = os.path.join(os.path.expanduser('~'), 'Documents/Websites/JavierParada.github.io/EL5M/')
print(out_dir)

/Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/


# Download list of countries (remove CHI)

In [3]:
df = wb.data.DataFrame('AG.LND.EL5M.ZS', wb.region.members('WLD'), time=2010, labels=True).reset_index()
df["economy"]= df["economy"].replace("XKX", "KSV") # replace code for Kosovo
df = df.sort_values(by=['Country'], ascending=True)
countries = list(df["economy"])
print('{} countries in WDI'.format(len(countries)))
df = df[df["economy"]!="CHI"]
df

217 countries in WDI


,economy,Country,AG.LND.EL5M.ZS
140,AFG,Afghanistan,NaN
107,ALB,Albania,4.861273
141,DZA,Algeria,0.025006
183,ASM,American Samoa,3.555492
48,AND,Andorra,NaN
...,...,...,...
26,VIR,Virgin Islands (U.S.),6.638373
41,PSE,West Bank and Gaza,0.094975
125,YEM,"Yemen, Rep.",0.414368
68,ZMB,Zambia,NaN


# Borders

In [4]:
# 2015 GAUL Subnational Boundaries (Modified for World Bank Offical Use)
GAUL_1_256 = 'shapes/GAUL_1_256.shp'
GAUL_2_3411 = 'shapes/GAUL_2_3411.shp'

GAUL_borders_1 = gpd.read_file(os.path.join(out_dir,GAUL_1_256))
# replace codes for Channel Islands
GAUL_borders_1["ISO3166_1_"]= GAUL_borders_1["ISO3166_1_"].replace("GGY", "CHI")
GAUL_borders_1["ISO3166_1_"]= GAUL_borders_1["ISO3166_1_"].replace("JEY", "CHI")

# GAUL_borders_2
GAUL_borders_2 = gpd.read_file(os.path.join(out_dir,GAUL_2_3411))

# Remove Caspian Sea
# 2528 147307 1554 3098 1723
GAUL_borders_2 = GAUL_borders_2[(GAUL_borders_2["ADM1_CODE"]!=2528) & (GAUL_borders_2["ADM1_CODE"]!=147307) & (GAUL_borders_2["ADM1_CODE"]!=1554) & (GAUL_borders_2["ADM1_CODE"]!=3098) & (GAUL_borders_2["ADM1_CODE"]!=1723)]

In [5]:
codes = GAUL_borders_1[["ADM0_NAME", "ISO3166_1_"]]
codes

,ADM0_NAME,ISO3166_1_
0,Mozambique,MOZ
1,Mauritius,MUS
2,Malawi,MWI
3,Rwanda,RWA
4,Somalia,SOM
...,...,...
251,Sint Maarten (Neth.),SXM
252,Saint-Barthélemy (Fr.),None
253,Saint-Martin (Fr.),MAF
254,Kosovo,KSV


In [6]:
# Merge ISO3166 codes because they are not available on GAUL_borders_2
borders = GAUL_borders_2.merge(codes, left_on='ADM0_NAME', right_on='ADM0_NAME', suffixes=('_left', '_right'))
print(borders.crs)

epsg:4326


# dictOfShapes

In [7]:
dictOfShapes = {}
for i, (iso, name) in enumerate(zip(df["economy"], df["Country"])):
    print(i+1, iso, name)
    
    # Save country shapes
    # lsoas = borders[borders["ISO3166_1_"] == iso]
    # lsoas.to_file("/Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/shapes/217/{}.shp".format(iso))
    
    # Plot country shapes
    #f, ax = plt.subplots(1, figsize=(6,6))
    #rgb = (random.random(), random.random(), random.random())
    #ax = lsoas.plot(ax=ax, color=rgb, edgecolor='black')
    #plt.title('Country {}: {} ({})'.format(i+1,name,iso))
    #plt.show()
    
    # Import country shapes into EE
    dictOfShapes["{0}".format(iso)] = geemap.shp_to_ee("/Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/shapes/217 simplified/{}.shp".format(iso), encoding='latin-1')

1 AFG Afghanistan
2 ALB Albania
3 DZA Algeria
4 ASM American Samoa
5 AND Andorra
6 AGO Angola
7 ATG Antigua and Barbuda
8 ARG Argentina
9 ARM Armenia
10 ABW Aruba
11 AUS Australia


KeyboardInterrupt: 

In [10]:
Map = geemap.Map(center = (40, -100), zoom = 3)
Map
#Map.addLayer(countries, {}, 'Countries')

Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=(T…

In [11]:
# Example:
pickCountry = 'VNM'
dictOfShapes[pickCountry]
# Add shapefile
Map.addLayer(dictOfShapes[pickCountry], {}, pickCountry)
Map.centerObject(dictOfShapes[pickCountry])

KeyError: 'VNM'

In [12]:
# Elevation 
srtm = ee.Image('CGIAR/SRTM90_V4').select('elevation')
merit = ee.Image("MERIT/DEM/v1_0_3").select('dem')

#.clip(dictOfShapes[pickCountry])

# Set visualization parameters for elevation
vis_params = {
  'min': 0,
  'max': 4000,
  'palette': ['006633', 'E5FFCC', '662A00', 'D8D8D8', 'F5F5F5']}

Map.addLayer(srtm, vis_params, 'SRTM Digital Elevation Data Version 4', True, opacity=1.0)
Map.addLayer(merit, vis_params, 'MERIT DEM', True, opacity=1.0)

In [13]:
borders_gaul = ee.FeatureCollection("FAO/GAUL/2015/level1")
Map.addLayer(borders_gaul, {}, 'Country Boundaries')

In [14]:
# Create a binary layer using logical operations.
lowAreas2 = srtm.lte(5)
lowAreas = merit.lte(5)

#.And(elevation.gte(0))
Map.addLayer(lowAreas2.selfMask(), {}, 'Low Areas SRTM<= 5 meters')
Map.addLayer(lowAreas.selfMask(), {}, 'Low Areas MERIT<= 5 meters')

In [11]:
# Run zonal statistics by shapefile
skip = ['AUS', 'BRA', 'IND', 'RUS']

#skip = ['MEX', 'AUS', 'BRA', 'ECU', 'CHL', 'CAN', 'UGA', 'ZAF', 'RUS', 'MDG','THA','IDN', 'IND', 'GNB', 'EST', 'USA']

for i, (iso, name) in enumerate(zip(df["economy"], df["Country"])):
    print(i+1, iso, name)
    nlcd_stats = os.path.join(out_dir, 'results/SUM_{}.csv'.format(iso))  
    if iso not in skip:
        try:
            geemap.zonal_statistics_by_group(lowAreas, dictOfShapes[iso], nlcd_stats, statistics_type='SUM', denominator=1000000, decimal_places=2)
        except:
            print("An exception occurred")
    else:
        print("Skipped")

1 AFG Afghanistan
Skipped
2 ALB Albania
Skipped
3 DZA Algeria
Skipped
4 ASM American Samoa
Skipped
5 AND Andorra
Skipped
6 AGO Angola
Skipped
7 ATG Antigua and Barbuda
Skipped
8 ARG Argentina
Skipped
9 ARM Armenia
Skipped
10 ABW Aruba
Skipped
11 AUS Australia
An exception occurred
12 AUT Austria
Skipped
13 AZE Azerbaijan
Skipped
14 BHS Bahamas, The
Skipped
15 BHR Bahrain
Skipped
16 BGD Bangladesh
Skipped
17 BRB Barbados
Skipped
18 BLR Belarus
Skipped
19 BEL Belgium
Skipped
20 BLZ Belize
Skipped
21 BEN Benin
Skipped
22 BMU Bermuda
Skipped
23 BTN Bhutan
Skipped
24 BOL Bolivia
Skipped
25 BIH Bosnia and Herzegovina
Skipped
26 BWA Botswana
Skipped
27 BRA Brazil
An exception occurred
28 VGB British Virgin Islands
Skipped
29 BRN Brunei Darussalam
Skipped
30 BGR Bulgaria
Skipped
31 BFA Burkina Faso
Skipped
32 BDI Burundi
Skipped
33 CPV Cabo Verde
Skipped
34 KHM Cambodia
Skipped
35 CMR Cameroon
Skipped
36 CAN Canada
Skipped
37 CYM Cayman Islands
Skipped
38 CAF Central African Republic
Skipped
3

# Australia

In [17]:
iso = 'AUS'
nlcd_stats = os.path.join(out_dir, 'results/SUM_{}.csv'.format(iso))  
#fc = geemap.geopandas_to_ee(borders[borders["ISO3166_1_"] == iso])
fc = geemap.shp_to_ee("shapes/217 simplified/AUS.shp")
#fc = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM0_NAME', 'Australia'))
geemap.zonal_statistics_by_group(lowAreas, fc, nlcd_stats, statistics_type='SUM', denominator=1000000, decimal_places=2)

Computing ... 
Generating URL ...
Please wait ...
Data downloaded to /Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/results/SUM_AUS.csv


# Brazil

In [34]:
iso = 'BRA'
nlcd_stats = os.path.join(out_dir, 'results/SUM_{}.csv'.format(iso))  
fc = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM0_NAME', 'Brazil'))
geemap.zonal_statistics_by_group(lowAreas, fc, nlcd_stats, statistics_type='SUM', denominator=1000000, decimal_places=2)

Computing ... 
Generating URL ...
Please wait ...
Data downloaded to /Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/results/SUM_BRA.csv


# India

In [35]:
iso = 'IND'
nlcd_stats = os.path.join(out_dir, 'results/SUM_{}.csv'.format(iso))  
fc = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM0_NAME', 'India'))
geemap.zonal_statistics_by_group(lowAreas, fc, nlcd_stats, statistics_type='SUM', denominator=1000000, decimal_places=2)

Computing ... 
Generating URL ...
Please wait ...
Data downloaded to /Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/results/SUM_IND.csv


# Russia

In [32]:
iso = 'RUS'
nlcd_stats = os.path.join(out_dir, 'results/SUM_{}.csv'.format(iso))  
fc = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM0_NAME', 'Russian Federation'))
geemap.zonal_statistics_by_group(lowAreas, fc, nlcd_stats, statistics_type='SUM', denominator=1000000, decimal_places=2)

Computing ... 
Generating URL ...
Please wait ...
Data downloaded to /Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/results/SUM_RUS.csv


## Append all results

In [18]:
# 215 countries
os.chdir("/Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/results/")
extension = 'csv'
all_filenames = [i for i in glob.glob('*.{}'.format(extension))]
len(all_filenames)

216

In [25]:
# allCountries
allCountries = pd.concat([pd.read_csv(f) for f in all_filenames])
allCountries["Percent_0"] = allCountries["Class_0"]/allCountries["Class_sum"]
allCountries["Percent_1"] = 1 - allCountries["Percent_0"]
allCountries = allCountries.sort_values(by=['Percent_1'], ascending=False)
# Add Australia to results from Australia
allCountries.loc[allCountries["ADM0_NAME"]=="Australia", "ISO3166_1_"] = "AUS"
allCountries.loc[allCountries["ADM0_NAME"]=="Brazil", "ISO3166_1_"] = "BRA"
allCountries.loc[allCountries["ADM0_NAME"]=="India", "ISO3166_1_"] = "IND"
allCountries.loc[allCountries["ADM0_NAME"]=="Russian Federation", "ISO3166_1_"] = "RUS"

allCountries.to_csv("../allCountries.csv", index=False, encoding ='latin-1')

In [26]:
national = allCountries.groupby("ISO3166_1_").sum()
national["Result_1"] = national["Class_1"]/national["Class_sum"]*100
national = national.sort_values(by=['Result_1'], ascending=False)
national.reset_index(inplace=True)
national = national.merge(codes, left_on='ISO3166_1_', right_on='ISO3166_1_', suffixes=('_left', '_right'))
national

,ISO3166_1_,Class_0,Class_sum,ADM1_CODE,EXP1_YEAR,Shape_Le_1,STR1_YEAR,OBJECTID,Shape_Leng,ADM0_CODE,Shape_Area,Class_1,AREASQKM16,STE_CODE16,Percent_0,Percent_1,Result_1,ADM0_NAME
0,BHS,3848.90,11803.69,561.0,3000.0,159.501236,1000.0,1046.0,159.501236,20.0,1.058404,7954.79,0.0,0.0,0.326076,0.673924,67.392400,The Bahamas
1,KIR,416.19,995.13,1739.0,3000.0,19.385129,1000.0,3385.0,19.385129,135.0,0.082954,578.94,0.0,0.0,0.418227,0.581773,58.177324,Kiribati
2,NLD,16583.78,35169.71,25962.0,36000.0,58.560087,12000.0,38694.0,58.560087,2124.0,4.632765,18585.93,0.0,0.0,4.801919,7.198081,52.846412,Netherlands
3,CYM,146.71,265.28,206229.0,9000.0,2.217021,3000.0,2991.0,2.217021,144.0,0.023098,118.57,0.0,0.0,1.886454,1.113546,44.696170,Cayman Islands (U.K.)
4,MDV,110.98,190.14,38303.0,60000.0,18.718868,20000.0,39990.0,18.718868,3080.0,0.018191,79.16,0.0,0.0,12.331275,7.668725,41.632481,Maldives
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,LAO,230368.38,230368.38,175094.0,51000.0,111.126769,20007.0,30838.0,111.126769,2363.0,19.723221,0.00,0.0,0.0,17.000000,0.000000,0.000000,Lao People's Democratic Republic
212,TCD,1265183.22,1265183.22,301484.0,84000.0,238.393820,51000.0,11298.0,238.393820,1400.0,106.690200,0.00,0.0,0.0,28.000000,0.000000,0.000000,Chad
213,NPL,147432.36,147432.36,10770.0,15000.0,50.184866,5000.0,10060.0,50.184866,875.0,13.559747,0.00,0.0,0.0,5.000000,0.000000,0.000000,Nepal
214,KGZ,199179.33,199179.33,305083.0,24000.0,94.747833,11998.0,12396.0,94.747833,1104.0,21.471030,0.00,0.0,0.0,8.000000,0.000000,0.000000,Kyrgyz Republic


In [27]:
fig = go.Figure(data=go.Choropleth(
    locations = national['ISO3166_1_'],
    z = national['Result_1'],
    text = national['ADM0_NAME'],
    colorscale = 'inferno',
    autocolorscale=False,
    reversescale=True,
    marker_line_color='darkgray',
    marker_line_width=0.5,
    colorbar_ticksuffix = '%',
    colorbar_title = 'AG.LND.EL5M.ZS<br>',
))

fig.update_layout(
    title_text='Land area where elevation is below 5 meters (% of total land area)',
    geo=dict(
        showframe=False,
        showcoastlines=False,
        projection_type='equirectangular'
    ),
    annotations = [dict(
        x=0.55,
        y=0.1,
        xref='paper',
        yref='paper',
        text='Source: <a href="https://databank.worldbank.org/source/world-development-indicators">\
            WDI</a>',
        showarrow = False
    )]
)

fig.show()
fig.write_html("../../EL5M_Area_MERIT.html")

end